# Criando as representações gráficas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os

Importando os dados

In [ ]:
corr_scores = pd.read_csv("data/corr.scores.csv")
exams = pd.read_csv("data/exams.csv")
mae_scores = pd.read_csv("data/mae.scores.csv")
r2_scores = pd.read_csv("data/r2.scores.csv")

Exames

In [ ]:
exams.head()

escores de correlação

In [ ]:
corr_scores.head()

Escores de MAE

In [ ]:
mae_scores.head()

R2 escores

In [ ]:
r2_scores.head()

Verificação se todos os exames são iguais

In [ ]:
print((mae_scores["exam_id"] == r2_scores["exam_id"]).all())
print((mae_scores["exam_id"] == corr_scores["exam_id"]).all())
print((r2_scores["exam_id"] == corr_scores["exam_id"]).all())

In [ ]:
ids_mae = set(mae_scores["exam_id"])
ids_r2 = set(r2_scores["exam_id"])
ids_corr = set(corr_scores["exam_id"])
ids_exams = set(exams["exam_id"])

print("MAE == R²:", ids_mae == ids_r2)
print("MAE == Corr:", ids_mae == ids_corr)
print("R² == Corr:", ids_r2 == ids_corr)

ids_comuns = ids_mae & ids_r2 & ids_corr

print("Exames em comum:", len(ids_comuns))
print("Exames em exams:", len(ids_exams))
print("Todos os exames das métricas estão em exams:",
      ids_comuns.issubset(ids_exams))

## Gerando tabela de métricas dos scores do modelo

In [ ]:
resultado = pd.DataFrame({
    "Derivação": mae_scores.columns,
    "MAE": mae_scores.mean().values,
    "R²": r2_scores.mean().values,
    "Correlação": corr_scores.mean().values
})

resultado = resultado.drop(
    resultado[resultado["Derivação"] == "exam_id"].index
)

resultado.to_csv("data/resultados/tabela_metricas_por_derivacao.csv", index=False)

resultado.head(13)

## Gerando tabela de métricas por tipo de problema (disease)

Verificação de consistência dos dados

In [ ]:
corr = corr_scores.drop(columns=["exam_id"]).mean(axis=1)
mae  = mae_scores.drop(columns=["exam_id"]).mean(axis=1)
r2   = r2_scores.drop(columns=["exam_id"]).mean(axis=1)

metrics = pd.DataFrame({
    "exam_id": corr_scores["exam_id"],
    "corr": corr,
    "mae": mae,
    "r2": r2
})

df = exams.merge(metrics, on="exam_id")

diseases = ["1dAVb", "RBBB", "LBBB", "SB", "ST", "AF"]
has_disease = df[diseases].any(axis=1)
inconsistent = df[(df["normal_ecg"] == True) & (has_disease == True)]
print("Casos inconsistentes:", len(inconsistent))
inconsistent.head()

In [ ]:
total = len(df)

n_normal = df["normal_ecg"].sum()
n_non_normal = (~df["normal_ecg"]).sum()

print("Total:", total)
print("Normal:", n_normal)
print("Não normal:", n_non_normal)
print("Soma:", n_normal + n_non_normal)

Gerando a tabela por tipo de problema

In [ ]:
diseases = ["normal_ecg", "1dAVb", "RBBB", "LBBB", "SB", "ST", "AF"]

# Agregação final por classe
result = []

for d in diseases:
    subset = df[df[d] == True]

    result.append({
        "disease": d,
        "corr_mean": subset["corr"].mean(),
        "mae_mean": subset["mae"].mean(),
        "r2_mean": subset["r2"].mean(),
        "n": len(subset)
    })

result = pd.DataFrame(result)

result.to_csv("data/resultados/tabela_metricas_por_doenca.csv", index=False)

result.head(7)

## Gerando boxplots

genero e faixa etária

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# 1. Recriação do age_group com base na idade numérica do DataFrame exams
bins = [15, 24, 34, 44, 54, 64, 110]
labels = ['15-24', '25-34', '35-44', '45-54', '55-64', '65+']

# Mapeia a idade numérica para as faixas de texto
exams['age_group'] = pd.cut(exams['age'], bins=bins, labels=labels, right=True)

# 2. Preparação dos dados (Média das derivações)
corr_scores['mean_val'] = corr_scores.drop(columns=['exam_id']).mean(axis=1)
mae_scores['mean_val'] = mae_scores.drop(columns=['exam_id']).mean(axis=1)
r2_scores['mean_val'] = r2_scores.drop(columns=['exam_id']).mean(axis=1)

def preparar_df(df_metric, nome_metrica):
    temp = df_metric[['exam_id', 'mean_val']].merge(exams[['exam_id', 'age_group', 'is_male']], on='exam_id')
    temp['metric'] = nome_metrica
    return temp

df_final = pd.concat([
    preparar_df(corr_scores, 'Correlation'),
    preparar_df(mae_scores, 'MAE'),
    preparar_df(r2_scores, 'R2')
])

df_final['gender'] = df_final['is_male'].map({True: 'Male', False: 'Female'})
ordem_idades = ['65+', '55-64', '45-54', '35-44', '25-34', '15-24']
df_final['age_group'] = pd.Categorical(df_final['age_group'], categories=ordem_idades, ordered=True)

# 2. Configuração do Layout Avançado
fig = plt.figure(figsize=(16, 6))
outer_grid = gridspec.GridSpec(1, 3, wspace=0.10)

metricas = ['Correlation', 'MAE', 'R2']
cores_fill = {'Male': '#9BBFDC', 'Female': '#EAA6B5'} 
cores_edge = {'Male': '#5B84A8', 'Female': '#C3687B'} 

limites_esquerda = {'Correlation': (1.0, 0.70), 'MAE': (0.40, 0.05), 'R2': (1.0, 0.50)}
limites_direita  = {'Correlation': (0.70, 1.0), 'MAE': (0.05, 0.40), 'R2': (0.50, 1.0)}

ax_primeiro_y = None

for i, metrica in enumerate(metricas):
    inner_grid = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer_grid[i], wspace=0.0)
    
    if i == 0:
        ax_male = plt.subplot(inner_grid[0])
        ax_primeiro_y = ax_male  
    else:
        ax_male = plt.subplot(inner_grid[0], sharey=ax_primeiro_y)
        
    ax_female = plt.subplot(inner_grid[1], sharey=ax_primeiro_y)
    
    # --- PLOT MALE (Esquerda) ---
    dados_male = df_final[(df_final['metric'] == metrica) & (df_final['gender'] == 'Male')]
    sns.boxplot(
        data=dados_male, x='mean_val', y='age_group', ax=ax_male, order=ordem_idades,
        showfliers=False, width=0.35, linewidth=1.2, color=cores_fill['Male'],
        boxprops={'edgecolor': cores_edge['Male']},
        whiskerprops={'color': cores_edge['Male']},
        capprops={'color': cores_edge['Male']},
        medianprops={'color': 'black', 'linewidth': 1.8}
    )
    ax_male.set_xlim(limites_esquerda[metrica])
    
    # --- PLOT FEMALE (Direita) ---
    dados_female = df_final[(df_final['metric'] == metrica) & (df_final['gender'] == 'Female')]
    sns.boxplot(
        data=dados_female, x='mean_val', y='age_group', ax=ax_female, order=ordem_idades,
        showfliers=False, width=0.35, linewidth=1.2, color=cores_fill['Female'],
        boxprops={'edgecolor': cores_edge['Female']},
        whiskerprops={'color': cores_edge['Female']},
        capprops={'color': cores_edge['Female']},
        medianprops={'color': 'black', 'linewidth': 1.8}
    )
    ax_female.set_xlim(limites_direita[metrica])
    
    # --- AJUSTE DE GRANULARIDADE PARA O GRÁFICO R2 ---
    if metrica == 'R2':
        ticks_r2 = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
        ax_male.set_xticks(ticks_r2)
        ax_female.set_xticks(ticks_r2)
    
    # --- CUSTOMIZAÇÃO ESTÉTICA ---
    for ax in [ax_male, ax_female]:
        ax.set_xlabel('')
        
        # 1. Ativa linhas verticais pontilhadas bem suaves ao fundo de cada número
        ax.grid(True, axis='x', linestyle='--', alpha=0.5, color='#D3D3D3', zorder=0)
        ax.set_axisbelow(True) # Garante que a grade fique atrás dos boxplots
        
        # 2. Ativa os tracinhos pretos (ticks) apontando para o número do eixo X
        ax.tick_params(axis='x', bottom=True, direction='out', colors='#222222', length=5, width=1.2)
        
        for spine in ['top', 'bottom', 'left', 'right']:
            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_color('#222222')
            ax.spines[spine].set_linewidth(1.2)
            
        if ax == ax_male:
            ax.spines['right'].set_linestyle((0, (5, 4)))
            if i == 0:
                ax.set_ylabel('Age Group', fontsize=11, fontweight='bold')
                ax.tick_params(axis='y', left=True, labelleft=True)
            else:
                ax.set_ylabel('')
                ax.tick_params(axis='y', left=False, labelleft=False)
        else:
            ax.spines['left'].set_visible(False)
            ax.set_ylabel('')
            ax.tick_params(axis='y', left=False, labelleft=False)
            
    ax_male.text(1.0, -0.18, metrica, transform=ax_male.transAxes, 
                 fontsize=12, fontweight='bold', ha='center')

# Legenda global na base
handles = [plt.Rectangle((0,0), 1, 1, color=cores_fill['Male'], ec=cores_edge['Male']), 
           plt.Rectangle((0,0), 1, 1, color=cores_fill['Female'], ec=cores_edge['Female'])]

fig.legend(handles, ['Male', 'Female'], loc='lower center', 
           bbox_to_anchor=(0.5, -0.05), ncol=2, frameon=True, 
           edgecolor='#CCCCCC', facecolor='white', fontsize=11,
           handlelength=1.5, handleheight=0.9)

plt.tight_layout()
fig.subplots_adjust(bottom=0.17)
plt.show()

In [ ]:
dados_female[dados_female["age_group"]=="15-24"].describe()

doenças (diseases)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Preparação dos dados no formato linear
# (assumindo que o DataFrame original 'df' já foi carregado)
df_lista = []

# Grupo de Controle (Normal)
df_normal = df[df["normal_ecg"] == True].copy()
df_normal["category"] = "Normal"
df_lista.append(df_normal)

# Patologias
diseases = ["1dAVb", "RBBB", "LBBB", "SB", "ST", "AF"]
for d in diseases:
    df_disease = df[df[d] == True].copy()
    df_disease["category"] = d
    df_lista.append(df_disease)

df_final_linear = pd.concat(df_lista, ignore_index=True)

# Definição estrita da ordem no Eixo Y
ordem_categorias = ["Normal"] + diseases
df_final_linear["category"] = pd.Categorical(df_final_linear["category"], categories=ordem_categorias, ordered=True)

# ==========================================================
# NOVO: Geração e Exportação do CSV com as métricas agregadas (Ajustado IQR)
# ==========================================================
# Agrupa pela patologia e calcula as estatísticas iniciais
df_estatisticas = df_final_linear.groupby('category')[['corr', 'mae', 'r2']].describe()

# Ajusta o 'min' e 'max' para ignorar outliers usando o método IQR (idêntico ao gráfico)
for metrica in ['corr', 'mae', 'r2']:
    for cat in ordem_categorias:
        serie = df_final_linear[df_final_linear['category'] == cat][metrica].dropna()
        if not serie.empty:
            q1 = serie.quantile(0.25)
            q3 = serie.quantile(0.75)
            iqr = q3 - q1
            limite_inferior = q1 - 1.5 * iqr
            limite_superior = q3 + 1.5 * iqr
            
            # Encontra os valores reais que formam as pontas dos "bigodes" no gráfico
            whisker_min = serie[serie >= limite_inferior].min()
            whisker_max = serie[serie <= limite_superior].max()
            
            # Substitui os valores de mínimo e máximo absolutos pelos valores sem outliers na tabela
            df_estatisticas.loc[cat, (metrica, 'min')] = whisker_min
            df_estatisticas.loc[cat, (metrica, 'max')] = whisker_max

# Achata as colunas (que ficam com níveis duplos devido ao describe) para facilitar a leitura no CSV
df_estatisticas.columns = [f"{metrica}_{estatistica}" for metrica, estatistica in df_estatisticas.columns]

# Transforma o índice (category) em uma coluna normal para alinhar o cabeçalho
df_estatisticas = df_estatisticas.reset_index()

# Define o nome do arquivo e salva o CSV no diretório local sem o índice extra
caminho_csv = 'estatisticas_metricas_por_patologia.csv'
df_estatisticas.to_csv(caminho_csv, index=False)
print(f"Sucesso! O arquivo com os resultados foi salvo de forma estruturada como: {caminho_csv}")
# ==========================================================

# 2. Renderização do Painel Acadêmico
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

# Mantendo o R2 formatado corretamente em negrito
metricas_colunas = {'Correlation': 'corr', 'MAE': 'mae', r'$\mathbf{R^2}$': 'r2'}
metricas = ['Correlation', 'MAE', r'$\mathbf{R^2}$']

cor_fill = '#72B2D1'   
cor_edge = '#437C99'   

# LIMITES AMPLIADOS PARA EVITAR CORTES NAS EXTREMIDADES
limites = {
    'Correlation': (0.65, 1.01),  
    'MAE': (0.02, 0.45),          
    r'$\mathbf{R^2}$': (0.45, 1.01)            
}

for i, metrica in enumerate(metricas):
    ax = axes[i]
    col_nome = metricas_colunas[metrica]
    
    sns.boxplot(
        data=df_final_linear, x=col_nome, y='category', ax=ax, order=ordem_categorias,
        showfliers=False, width=0.4, linewidth=1.2, color=cor_fill,
        boxprops={'edgecolor': cor_edge},
        whiskerprops={'color': cor_edge},
        capprops={'color': cor_edge},
        medianprops={'color': 'black', 'linewidth': 1.8}
    )
    
    ax.set_xlim(limites[metrica])
    ax.set_xlabel(metrica, fontsize=12, fontweight='bold', labelpad=10)
    
    # 1. Ativa linhas verticais pontilhadas bem suaves ao fundo de cada número
    ax.grid(True, axis='x', linestyle='--', alpha=0.5, color='#D3D3D3', zorder=0)
    ax.set_axisbelow(True) # Garante que a grade fique atrás dos boxplots
    
    # 2. Ativa os tracinhos pretos (ticks) apontando para os números do eixo X
    ax.tick_params(axis='x', bottom=True, direction='out', colors='#222222', length=5, width=1.2)
    
    # Moldura preta sólida estilo artigo
    for spine in ['top', 'bottom', 'left', 'right']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color('#222222')
        ax.spines[spine].set_linewidth(1.2)
        
    if i == 0:
        ax.set_ylabel('Condition', fontsize=12, fontweight='bold')
        ax.tick_params(axis='y', labelsize=11)
    else:
        ax.set_ylabel('')
        ax.tick_params(left=False)

plt.tight_layout()
plt.show()

bloxplot das derivações

In [ ]:
# não consigo fazer isso pq preciso de todos os dados dos exames em si

## Plotando 5 melhores e 5 piores ecgs

Encontrando os 5 melhores ECGs (com base no maior R²)

In [ ]:
# 1. Calcula a média do R2 entre todas as derivações para cada exame
colunas_sinal = [col for col in r2_scores.columns if col != 'exam_id' and col != 'mean_val']
r2_scores['mean_val'] = r2_scores[colunas_sinal].mean(axis=1)

# 2. Ordena o DataFrame do maior R2 para o menor
# 'ascending=False' garante que os maiores valores fiquem no topo
r2_ordenado = r2_scores.sort_values(by='mean_val', ascending=False)

# 3. Determina os 5 exames com os maiores R2
top_5_maiores_r2 = r2_ordenado.head(5)

# Exibe o ID do exame e a média do R2 alcançada
print("--- 5 Exames com Maior R2 (Melhores Reconstruções) ---")
print(top_5_maiores_r2[['exam_id', 'mean_val']].to_string(index=False))

Encontrando os 5 piores ECGs (com base no menor R²)

In [ ]:
# 1. Garante que a média do R2 está calculada (sem contar as colunas de ID e médias antigas)
colunas_sinal = [col for col in r2_scores.columns if col != 'exam_id' and col != 'mean_val']
r2_scores['mean_val'] = r2_scores[colunas_sinal].mean(axis=1)

# 2. Ordena o DataFrame do menor R2 para o maior
# 'ascending=True' coloca as piores reconstruções no topo
r2_piores_ordenado = r2_scores.sort_values(by='mean_val', ascending=True)

# 3. Determina os 5 exames com os menores R2
top_5_piores_r2 = r2_piores_ordenado.head(5)

# Exibe o ID do exame e a média do R2 alcançada
print("--- 5 Exames com Menor R2 (Piores Reconstructions) ---")
print(top_5_piores_r2[['exam_id', 'mean_val']].to_string(index=False))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Caminhos base das pastas
base_path = "data/top_ecgs"
best_dir = os.path.join(base_path, "best_ecgs")
worst_dir = os.path.join(base_path, "worst_ecgs")

# --- DETERMINAÇÃO DO RANKING AUTOMÁTICO PELO R² ---
colunas_sinal = [col for col in r2_scores.columns if col not in ['exam_id', 'mean_val']]

# Garante que a média atualizada está calculada para ordenação de ranking
r2_scores['mean_val'] = r2_scores[colunas_sinal].mean(axis=1)
corr_scores['mean_val'] = corr_scores[colunas_sinal].mean(axis=1)

# Ordena e extrai os IDs convertidos para string (para bater com os nomes dos arquivos)
best_ids_ranked = r2_scores.sort_values(by='mean_val', ascending=False).head(5)['exam_id'].astype(str).tolist()
worst_ids_ranked = r2_scores.sort_values(by='mean_val', ascending=True).head(5)['exam_id'].astype(str).tolist()


def plot_paciente_duas_colunas_com_scores(ecg_id, directory, prefix, color_rec, idx_rank, fs=400):
    """
    Gera uma figura exclusiva por paciente com layout de 2 colunas por linha.
    Insere os valores de R e R² correspondentes a cada derivação na legenda e salva em PNG.
    """
    # 1. Carrega os dados de sinal
    df_orig = pd.read_csv(os.path.join(directory, f"{prefix} - ECG {ecg_id} - Original.csv"))
    df_rec = pd.read_csv(os.path.join(directory, f"{prefix} - ECG {ecg_id} - Reconstructed.csv"))
    
    if 'Unnamed: 0' in df_orig.columns: df_orig = df_orig.drop(columns=['Unnamed: 0'])
    if 'Unnamed: 0' in df_rec.columns: df_rec = df_rec.drop(columns=['Unnamed: 0'])
    
    derivacoes = df_orig.columns.tolist()
    num_derivacoes = len(derivacoes)
    num_linhas = int(np.ceil(num_derivacoes / 2))
    tempo = np.arange(len(df_orig)) / fs
    
    # Prepara a busca do ID correto nos DataFrames de scores
    try:
        id_busca = int(ecg_id)
    except ValueError:
        id_busca = str(ecg_id)

    # Cria o grid de subplots
    fig, axes = plt.subplots(num_linhas, 2, figsize=(16, 3 * num_linhas))
    axes = axes.flatten()
    
    for idx, col in enumerate(derivacoes):
        ax = axes[idx]
        
        # 2. Busca o score específico para ESTA derivação atual (col)
        try:
            val_corr = corr_scores.loc[corr_scores['exam_id'] == id_busca, col].values[0]
            val_r2 = r2_scores.loc[r2_scores['exam_id'] == id_busca, col].values[0]
            label_reconstrucao = f'Model Reconstruction (R = {val_corr:.3f}, $R^2$ = {val_r2:.3f})'
        except Exception:
            # Caso a coluna específica não exista, faz o fallback para a média global do exame
            val_corr = corr_scores.loc[corr_scores['exam_id'] == id_busca, 'mean_val'].values[0]
            val_r2 = r2_scores.loc[r2_scores['exam_id'] == id_busca, 'mean_val'].values[0]
            label_reconstrucao = f'Model Reconstruction (R = {val_corr:.3f}, $R^2$ = {val_r2:.3f})'
            
        # Plot dos sinais sobrepostos
        ax.plot(tempo, df_orig[col], color='#111111', label='Reference', linewidth=1.2)
        ax.plot(tempo, df_rec[col], color=color_rec, label=label_reconstrucao, linewidth=1.1, alpha=0.9)
        
        # Estética do artigo de referência
        ax.set_xlabel('Time (sec)', fontsize=10)
        ax.set_ylabel('Amplitude [$\mu$V]', fontsize=10)
        ax.set_title(f"Lead: {col}", fontsize=11, fontweight='bold', loc='left')
        
        ax.tick_params(axis='both', which='major', labelsize=9, direction='in', length=5, width=1.0)
        ax.grid(False)
        
        for spine in ['top', 'bottom', 'left', 'right']:
            ax.spines[spine].set_color('#222222')
            ax.spines[spine].set_linewidth(1.1)
            
        # Legenda com fundo branco e borda sutil
        ax.legend(loc='upper right', fontsize=8.5, frameon=True, facecolor='white', edgecolor='#CCCCCC')
        
    if num_derivacoes % 2 != 0:
        for i in range(num_derivacoes, len(axes)):
            axes[i].axis('off')
            
    # Busca a média do R² deste paciente para colocar no título principal do painel
    media_r2_paciente = r2_scores.loc[r2_scores['exam_id'] == id_busca, 'mean_val'].values[0]
    
    plt.suptitle(f"ECG Signal | Patient ID: {ecg_id} ({prefix} Rank #{idx_rank+1} - Mean $R^2$: {media_r2_paciente:.3f})", 
                 fontsize=15, fontweight='bold', y=1.01)
    
    plt.tight_layout()
    
    # --- SALVAMENTO AUTOMÁTICO ---
    # Define o nome do arquivo de forma organizada (ex: rank_1_BEST_1155295.png)
    nome_arquivo = f"rank_{idx_rank+1}_{prefix}_{ecg_id}.png"
    plt.savefig(nome_arquivo, bbox_inches='tight', dpi=300)
    print(f"Salvo com sucesso: {nome_arquivo}")
    
    plt.show()
    plt.close(fig) # Fecha a figura para liberar memória do Jupyter Notebook

# --- EXECUÇÃO EM ORDEM DE RANKING ---

print("Gerando e salvando painéis para os MELHORES...")
for idx_rank, ecg_id in enumerate(best_ids_ranked):
    plot_paciente_duas_colunas_com_scores(ecg_id, best_dir, "BEST", '#0055ff', idx_rank)

print("\nGerando e salvando painéis para os PIORES...")
for idx_rank, ecg_id in enumerate(worst_ids_ranked):
    plot_paciente_duas_colunas_com_scores(ecg_id, worst_dir, "WORST", '#e02424', idx_rank)

## Cálculo de métricas globais

In [ ]:
BASE_DIR = os.getcwd()

caminho_entrada = os.path.join(BASE_DIR, "data", "resultados", "tabela_metricas_por_derivacao.csv")
caminho_saida = os.path.join(BASE_DIR, "data", "resultados", "metricas_globais.csv")

df = pd.read_csv(caminho_entrada)


metricas_globais = pd.DataFrame(
    {
        "Metrica": ["MAE", "R²", "Correlação"],
        "Media_Global": [
            df["MAE"].mean(),
            df["R²"].mean(),
            df["Correlação"].mean(),
        ],
        "Std_Global": [df["MAE"].std(), df["R²"].std(), df["Correlação"].std()],
    }
)

metricas_globais.to_csv(caminho_saida, index=False)

print(f"Sucesso! Arquivo 'metricas_globais.csv' gerado em:\n{caminho_saida}")